# Train the learned temporal router

This notebook trains the **round-4 learned router** used in Adaptive Temporal RAG: a multitask DeBERTa classifier that decides, for each question,

1. whether retrieval is needed, and
2. which temporal class it belongs to (`current`, `historical`, or `timeless`).

The published r4 models were trained on year-rule-corrected LLM labels (80,145 / 10,018 / 9,901 train / val / test). On the frozen Checkpoint B evaluation they **tie** the rule-based Adaptive system: templated exact match 0.948, held-out exact match 0.762, TimeQA containment 0.296, with 0/0 discordant answers and McNemar *p* = 1 on all three sets. The learned router does not beat the rules; it matches them without a trivia whitelist, and is better calibrated than the earlier 100k router.

Weights and the full training JSONL files are not in git (they are larger than GitHub's file limit). Point `DATA_DIR` at a local or Drive copy of `train.jsonl`, `val.jsonl`, and `test.jsonl` to retrain. If the data is missing, the notebook still loads the committed result files and prints the published table.

Runtime: a CUDA GPU (the published run used an NVIDIA RTX PRO 6000 Blackwell Server Edition). A100, L4, or similar also work. Training time will differ by GPU.


## 1. Setup

Paths are parameters. Change them if your clone, Drive folder, or data zip lives somewhere else. No API keys are required: training is a local fine-tune of DeBERTa.


In [ ]:
import os
import sys
import shutil
import subprocess
import zipfile
from pathlib import Path

# --- parameters (edit these) ---
REPO_URL = "https://github.com/dakshagarwal76/adaptive-temporal-rag-project.git"
USE_DRIVE = True          # mount Google Drive when running on Colab
TRAIN_BASE = True         # fine-tune microsoft/deberta-v3-base
TRAIN_LARGE = True        # fine-tune microsoft/deberta-v3-large
SMOKE = False             # True: tiny subsample to test the pipeline quickly
APPLY_YEAR_RULE = True    # apply the explicit past-year historical rule before training
# Leave DATA_DIR / DATA_ZIP at None to auto-detect a clone or a Drive zip.
DATA_DIR = None           # folder containing train.jsonl, val.jsonl, test.jsonl
DATA_ZIP = None           # optional zip that contains those files (possibly under data/)
# -------------------------------

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "180"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

HERE = Path.cwd()


def looks_like_repo(path: Path) -> bool:
    return (path / "src" / "query_analyzer.py").is_file() and (path / "router_v2" / "train_v2.py").is_file()


def find_repo() -> Path:
    for cand in (
        HERE,
        HERE.parent,
        Path("/content/adaptive-temporal-rag-project/codebase/adaptive-temporal-rag"),
        Path("/content/adaptive-temporal-rag"),
        HERE / "codebase" / "adaptive-temporal-rag",
        Path(".."),
    ):
        if looks_like_repo(cand.resolve()):
            return cand.resolve()
    if IN_COLAB:
        clone_dir = Path("/content/adaptive-temporal-rag-project")
        if not (clone_dir / ".git").exists():
            subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)])
        repo = clone_dir / "codebase" / "adaptive-temporal-rag"
        if looks_like_repo(repo):
            return repo
    raise FileNotFoundError(
        "Could not find the research repo (src/query_analyzer.py and router_v2/train_v2.py). "
        "Open the notebook from a clone, or set REPO_URL and run on Colab so it can be cloned."
    )


REPO = find_repo()
RV2 = REPO / "router_v2"
OUT_DIR = Path("/content/router_v2_outputs") if IN_COLAB else (RV2 / "bundles" / "colab_run")
if IN_COLAB and USE_DRIVE:
    drive_out = Path("/content/drive/MyDrive/adaptive-temporal-rag/router_v2_outputs")
    drive_out.parent.mkdir(parents=True, exist_ok=True)
    OUT_DIR = drive_out
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(RV2))
print("IN_COLAB", IN_COLAB)
print("REPO", REPO)
print("OUT_DIR", OUT_DIR)


In [ ]:
import importlib, subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install("transformers>=4.46,<5", "sentencepiece", "accelerate", "tiktoken", "protobuf")

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU", torch.cuda.get_device_name(0))
    print("bf16_supported", torch.cuda.is_bf16_supported())
else:
    print("No CUDA GPU. Training will be very slow; the published numbers below still load from the committed result files.")


## 2. Data loading

The published split sizes are **80,145 / 10,018 / 9,901**. Those JSONL files are not committed. Place them in `DATA_DIR`, or set `DATA_ZIP` to an archive that contains them (a `data/` prefix inside the zip is fine).


In [ ]:
import json
from collections import Counter

PROBE_PATH = RV2 / "data" / "probe36.json"
CHALLENGE_PATH = RV2 / "data" / "challenge_v2.jsonl"


def unpack_zip(zip_path: Path, dest: Path) -> Path:
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest)
    for cand in (dest, dest / "data", *dest.glob("**/train.jsonl")):
        folder = cand if cand.is_dir() else cand.parent
        if (folder / "train.jsonl").is_file():
            return folder
    raise FileNotFoundError(f"No train.jsonl inside {zip_path}")


def autodetect_data() -> Path | None:
    candidates = []
    if DATA_DIR:
        candidates.append(Path(DATA_DIR))
    if DATA_ZIP:
        candidates.append(unpack_zip(Path(DATA_ZIP), Path("/content/data_r4") if IN_COLAB else (OUT_DIR / "data_unpacked")))
    candidates.extend([
        RV2 / "data",
        Path("/content/data_r4"),
        Path("/content/rv2/data"),
    ])
    if IN_COLAB:
        from glob import glob
        for pattern in (
            "/content/drive/MyDrive/**/train.jsonl",
            "/content/drive/Shareddrives/**/train.jsonl",
            "/content/drive/MyDrive/**/router_v2_r4_inputs.zip",
            "/content/drive/MyDrive/**/router_v2_colab_inputs.zip",
        ):
            for hit in glob(pattern, recursive=True):
                p = Path(hit)
                candidates.append(p.parent if p.suffix == ".jsonl" else unpack_zip(p, Path("/content/data_r4")))
    for folder in candidates:
        if folder and (Path(folder) / "train.jsonl").is_file():
            return Path(folder)
    return None


DATA = autodetect_data()
splits = {}
if DATA is None:
    print("Training JSONL not found. Training cells will be skipped; published results still load from router_v2/results/.")
else:
    for name in ("train", "val", "test"):
        path = DATA / f"{name}.jsonl"
        rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
        splits[name] = rows
        cells_c = Counter((r.get("temporal_class"), r.get("needs_retrieval"), r.get("origin")) for r in rows)
        print(f"{name}: {len(rows):,} rows  (path {path})")
    print("expected published sizes: train 80,145  val 10,018  test 9,901")

probe36 = json.loads(PROBE_PATH.read_text(encoding="utf-8")) if PROBE_PATH.exists() else []
challenge = [json.loads(line) for line in open(CHALLENGE_PATH, encoding="utf-8")] if CHALLENGE_PATH.exists() else []
print("eval-only sets  probe36", len(probe36), "challenge_v2", len(challenge))


## 3. Year-rule labelling

Round 4 exists because the Qwen2.5-7B temporal labels treated many questions with an explicit past year (often SituatedQA-style “as of YEAR”, 2015–2021) as `current`. The year rule is:

> A **real** question anchored to an explicit past year ≤ 2024 (November 2026 minus two years) is labelled `historical`. Years used as quantities or maths operands are ignored.

On the published data this changed **3,810 training labels** (4,855 across train / val / test: 3,520 + 513 + 465 `current→historical` and 290 + 29 + 38 `timeless→historical`). A further **274** leak-free recent-year office-holder pairs were appended to train (`aug_r4.py`). The necessity definition and this year rule were chosen after seeing evaluation failures; that should be disclosed with the result.


In [ ]:
import re
import collections

# Same masks as router_v2/build_v2_dataset.py and router_v2/fix_year_labels.py
UNIT = r"(?:seconds?|minutes?|hours?|days?|weeks?|grams?|kilograms?|kg|km|metres?|meters?|miles?|feet|inches|litres?|liters?|calories|dollars?|pounds?|euros?|people|units|points|pages|words|bc|bce)"
QTY_RE = re.compile(rf"\b(1[0-9]{{3}}|20[0-9]{{2}})\s+{UNIT}\b", re.I)
MATH_RE = re.compile(
    r"(?:(?:square root of|cube root of|plus|minus|times|divided by|multiplied by|sum of|product of)\s+(1[0-9]{3}|20[0-9]{2})\b)"
    r"|(?:\bis\s+(1[0-9]{3}|20[0-9]{2})\s+(?:a prime|an? (?:even|odd|prime)|divisible))",
    re.I,
)
YEAR = re.compile(r"\b(1[0-9]{3}|20[0-9]{2})\b")
REFERENCE_CUTOFF = 2024  # November 2026 minus 2 years


def anchor_years(question: str):
    masked = QTY_RE.sub(" ", question)
    masked = MATH_RE.sub(" ", masked)
    return [int(y) for y in YEAR.findall(masked)]


def apply_year_rule(row: dict, cutoff: int = REFERENCE_CUTOFF) -> dict:
    out = dict(row)
    if out.get("origin") == "real":
        years = anchor_years(out["query"])
        if years and max(years) <= cutoff and out.get("llm_temporal") != "historical":
            out["llm_temporal_orig"] = out.get("llm_temporal")
            out["llm_temporal"] = "historical"
    return out


examples = [
    "Who was the prime minister of the United Kingdom as of 2018?",
    "Who is the prime minister of the United Kingdom?",
    "Is 2017 a prime number?",
    "How many grams are in 2019 kg?",
]
print("Illustrative anchors (not a scored eval set):")
for text in examples:
    print(f"  {anchor_years(text)!r:20}  {text}")

if splits and APPLY_YEAR_RULE:
    change = collections.Counter()
    already = collections.Counter()
    for name, rows in splits.items():
        fixed = []
        for row in rows:
            # already applied in the published JSONL (llm_temporal_orig is set)
            if row.get("llm_temporal_orig"):
                already[f"{name}:{row['llm_temporal_orig']}->historical"] += 1
                fixed.append(row)
                continue
            new = apply_year_rule(row)
            if new.get("llm_temporal_orig"):
                change[f"{name}:{new['llm_temporal_orig']}->historical"] += 1
            fixed.append(new)
        splits[name] = fixed
    print("already present in the files (llm_temporal_orig):", dict(already) or "none")
    print("applied in this session:", dict(change) or "none")
    print("train size after rule", len(splits["train"]))
else:
    print("Year-rule function is defined. Load DATA_DIR and set APPLY_YEAR_RULE=True to apply it to a split.")


## 4. Training

The trainer is `router_v2/train_v2.py`. It fine-tunes `MultitaskQueryAnalyzer` with class-balanced sampling over `(temporal, needs)` cells and a confidence head trained with BCE on whether both heads are correct. After training it fits a temperature on the validation temporal logits, then writes `analyzer.pt`, `analyzer_meta.json`, and `eval_report_v2.json` (val / test / probe36 / challenge).

Published hyperparameters and wall-clock **training-loop** time from `analyzer_meta.json`:

| Model | Encoder | Epochs | Batch | Learning rate | Train *n* | Training time | GPU |
|-------|---------|-------:|------:|--------------:|----------:|--------------:|-----|
| r4_base | `microsoft/deberta-v3-base` | 3 | 64 | 3e-5 | 80,145 | 171.4 s (2 min 51 s) | NVIDIA RTX PRO 6000 Blackwell Server Edition |
| r4_large | `microsoft/deberta-v3-large` | 2 | 32 | 1e-5 | 80,145 | 406.5 s (6 min 47 s) | NVIDIA RTX PRO 6000 Blackwell Server Edition |

Temperature fitted on the published run: 1.6 (base), 1.4 (large). Set `SMOKE = True` to train on a small subsample first. Set `TRAIN_BASE` / `TRAIN_LARGE` to `False` to skip a model.


In [ ]:
def write_split_dir(rows_by_split, dest: Path) -> Path:
    dest.mkdir(parents=True, exist_ok=True)
    for name, rows in rows_by_split.items():
        with open(dest / f"{name}.jsonl", "w", encoding="utf-8") as handle:
            for row in rows:
                handle.write(json.dumps(row) + "\n")
    for extra in ("probe36.json", "challenge_v2.jsonl"):
        src = RV2 / "data" / extra
        if src.exists() and not (dest / extra).exists():
            shutil.copy2(src, dest / extra)
    return dest


def maybe_smoke(rows_by_split):
    if not SMOKE:
        return rows_by_split
    import random
    rng = random.Random(13)
    slim = {}
    slim["train"] = rng.sample(rows_by_split["train"], min(512, len(rows_by_split["train"])))
    slim["val"] = rng.sample(rows_by_split["val"], min(128, len(rows_by_split["val"])))
    slim["test"] = rng.sample(rows_by_split["test"], min(128, len(rows_by_split["test"])))
    print("SMOKE caps:", {k: len(v) for k, v in slim.items()})
    return slim


def train_router(tag: str, model_name: str, epochs: float, batch_size: int, lr: float) -> Path | None:
    if not splits:
        print(f"skip {tag}: no training data")
        return None
    if not torch.cuda.is_available() and not SMOKE:
        print(f"skip {tag}: no GPU (set SMOKE=True to try a tiny CPU run)")
        return None
    data_dir = write_split_dir(maybe_smoke(splits), OUT_DIR / "train_data")
    out = OUT_DIR / tag
    out.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, str(RV2 / "train_v2.py"),
        "--repo", str(REPO),
        "--data", str(data_dir),
        "--out", str(out),
        "--model", model_name,
        "--epochs", str(epochs),
        "--bs", str(batch_size),
        "--lr", str(lr),
        "--use-llm-labels",
    ]
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        cmd.append("--bf16")
    print("running", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)
    print("wrote", out)
    return out


print("TRAIN_BASE", TRAIN_BASE, "TRAIN_LARGE", TRAIN_LARGE, "SMOKE", SMOKE)


In [ ]:
base_dir = large_dir = None
if TRAIN_BASE:
    base_dir = train_router("r4_base", "microsoft/deberta-v3-base", epochs=3, batch_size=64, lr=3e-5)
if TRAIN_LARGE:
    large_dir = train_router("r4_large", "microsoft/deberta-v3-large", epochs=2, batch_size=32, lr=1e-5)
print("base_dir", base_dir)
print("large_dir", large_dir)


## 5. Bundle evaluation

`train_v2.py` already scores the validation and test JSONL plus the committed probe36 (36 questions) and challenge set (72 questions). The cell below prints that report for a just-trained bundle, or falls back to the committed `eval_report_v2.json` from the published run.

These probe and challenge figures are **router-head accuracy**, not retrieval exact match. Retrieval numbers are in the next section.


In [ ]:
from IPython.display import display, Markdown

ID2LABEL = {0: "current", 1: "historical", 2: "timeless"}


def both_correct_from_rows(rows):
    return sum(
        row["pred_temporal"] == row["gold_temporal"] and row["pred_needs"] == row["gold_needs"]
        for row in rows
    )


def show_bundle_report(label: str, path: Path):
    report = json.loads(path.read_text(encoding="utf-8"))
    lines = [f"### {label}", ""]
    lines.append("| Split | *n* | Temporal acc. | Needs acc. | Both correct | ECE (conf. head vs both correct) |")
    lines.append("|-------|----:|--------------:|-----------:|-------------:|---------------------------------:|")
    for split, title in (("val", "val"), ("test", "test"), ("probe36", "probe36"), ("challenge_v2", "challenge")):
        block = report[split]
        if split in ("probe36", "challenge_v2") and f"{split}_rows" in report:
            both = both_correct_from_rows(report[f"{split}_rows"])
            both_s = f"{both}/{block['n']}"
        else:
            both_s = "—"
        lines.append(
            f"| {title} | {block['n']} | {block['temporal_acc']:.3f} | {block['needs_acc']:.3f} "
            f"| {both_s} | {block['ece_conf_head_vs_both_correct']:.4f} |"
        )
    display(Markdown("\n".join(lines)))
    return report


committed = {
    "r4_base (published bundle report)": RV2 / "bundles" / "r4_base_fix" / "eval_report_v2.json",
    "r4_large (published bundle report)": RV2 / "bundles" / "r4_large_fix" / "eval_report_v2.json",
}
fresh = {}
if base_dir and (Path(base_dir) / "eval_report_v2.json").exists():
    fresh["r4_base (this run)"] = Path(base_dir) / "eval_report_v2.json"
if large_dir and (Path(large_dir) / "eval_report_v2.json").exists():
    fresh["r4_large (this run)"] = Path(large_dir) / "eval_report_v2.json"

for label, path in {**committed, **fresh}.items():
    if path.exists():
        show_bundle_report(label, path)
    else:
        print("missing", path)


## 6. Frozen retrieval evaluation (published)

The table below is **not** recomputed in this notebook. It is read from the committed files under `router_v2/results/`, which record a full run of the frozen Checkpoint B pipeline (`eval_router_v2.py`) with the published r4 bundles. Exact match is reported on the templated synthetic test (n = 77) and the six-entity held-out test (n = 193). TimeQA uses containment (n = 500), because exact match is near zero for every system on that subset.

Display rounding: 0.9480519… → 0.948, 0.7616580… → 0.762, TimeQA 0.296, ECE 0.0559357… → 0.0559 and 0.0477946… → 0.0478. Discordant counts are against the frozen rule-based Adaptive system.


In [ ]:
import pandas as pd

summary_path = RV2 / "results" / "published_r4_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))


def vs(block, key):
    x = block["vs_frozen_adaptive"][key]
    return x["mean_a"], x["n_a_only"], x["n_b_only"], x["mcnemar_p"]


rows = []
for tag, display in (("r4_base", "r4_base"), ("r4_large", "r4_large")):
    block = summary[tag]
    syn = vs(block, "synthetic_em")
    held = vs(block, "heldout_em")
    tqa = vs(block, "timeqa_contains")
    pr = block["probe36"]
    ch = block["challenge_v2"]
    ece = block["ece_conf_head_vs_both_correct_test"]
    rows.append({
        "Model": display,
        "Templated EM": f"{syn[0]:.3f}",
        "Held-out EM": f"{held[0]:.3f}",
        "TimeQA Contains": f"{tqa[0]:.3f}",
        "Discordant (a/b)": f"{syn[1]}/{syn[2]}, {held[1]}/{held[2]}, {tqa[1]}/{tqa[2]}",
        "McNemar p": f"{syn[3]:.0f}",
        "probe36 both": f"{pr['both_correct']}/{pr['n']} (temporal {pr['temporal_correct']})",
        "challenge both": f"{ch['both_correct']}/{ch['n']}",
        "Test ECE": f"{ece:.4f}",
        "Train seconds": f"{block['training']['train_seconds']:.1f}",
    })

frame = pd.DataFrame(rows)
display(frame)
display(Markdown(
    "The learned router **ties** the frozen Adaptive rules on all three retrieval evals "
    "(0/0 discordant, *p* = 1). r4_base probe36 is 29/36 both-correct (temporal 32/36); "
    "r4_large is 29/36 both-correct (temporal 33/36). Challenge-set both-correct is 65/72 (base) "
    "and 66/72 (large). Test-set ECE of the confidence head versus both heads correct is "
    "0.0559 (base) and 0.0478 (large)."
))


## Notes and caveats

- Full retrieval evaluation (`router_v2/eval_router_v2.py` and `run_eval_pipeline.sh`) needs the frozen corpus, FAISS index, and Checkpoint B config in this repo. It is intentionally not run from **Runtime → Run all**.
- Training checkpoints (`analyzer.pt`) and the 80k-row JSONL splits are not in git. The committed bundles include `analyzer_meta.json`, `eval_report_v2.json`, and a SHA-256 of the weight file.
- Disclose with the result: the necessity definition and the year rule were chosen after seeing evaluation failures; the authors’ tooling wrote the challenge set and the augmentation; the Qwen2.5-7B labels have no human check; probe36 is small.
- The earlier v1 100k router, without this year rule and without the whitelist, did not match the rules (templated EM 0.740, McNemar *p* = 3.05e-5). r4 removes that gap.
